# 14 · DBSCAN: grouping attackers who behave alike

DBSCAN is an **unsupervised clustering** model: it puts points that lie close together in one group and leaves isolated points as "noise". It does not need to be told how many groups there are.

Question: if we describe every flagged session (one IP) by **what it did**, does DBSCAN put attackers that did the same thing in the same group, even when they come from different IPs and from different sites?

- **Profile of a session:** the share of its flagged requests that belong to each step (scan, enum, bruteforce, sqli, xss, traversal, cmdi), exactly the steps the next-step model uses. Shares, not counts, so a long attack and a short one of the same kind look alike.
- **Truth:** the attack kinds the generator labelled for that session (campaign sessions have several).
- **Baseline:** group sessions by their main step (the step with the largest share).
- **Score:** adjusted Rand index (ARI): 1 = the groups match the truth exactly, 0 = no better than chance.

Data: development runs only (run3, run4 and camp7, both sites). The locked runs are not used.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from eval.final_test import run
from ml.next_step import STEPS, _BEHAVIOUR_STEP
from ml.train_defense import DEV_RUNS

D = ROOT / 'data'
NAMES = ['run3 JS', 'run4 JS', 'run4 DVWA', 'camp7 JS', 'camp7 DVWA']
parts = []
for name, (log, labels) in zip(NAMES, DEV_RUNS):
    req, sessions, report = run(D / 'raw' / log, D / 'labels' / labels)
    f = req[(req['pred'] == 1) | (req['session_attack'] == 1)].copy()
    f['step'] = [t if p == 1 else _BEHAVIOUR_STEP.get(b, '')
                 for p, b, t in zip(f['pred'], f['behaviour'], f['type_pred'])]
    f = f[f['step'] != '']
    profile = pd.crosstab(f['session'], f['step'], normalize='index').reindex(columns=STEPS, fill_value=0)
    truth = req[req['attack_type'] != ''].groupby('session')['attack_type'].agg(lambda s: '+'.join(sorted(set(s))))
    profile['truth'] = truth.reindex(profile.index).fillna('normal')
    profile['run'] = name
    profile['site'] = 'DVWA' if 'DVWA' in name else 'Juice Shop'
    parts.append(profile)
    print(name, report)
T = pd.concat(parts)
print(len(T), 'flagged sessions;', int((T['truth'] == 'normal').sum()), 'of them normal')
T['truth'].value_counts().head(10)

run3 JS {'log_lines': 4397, 'labels': 4397, 'matched': 4397, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


run4 JS {'log_lines': 4296, 'labels': 4296, 'matched': 4296, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


run4 DVWA {'log_lines': 3586, 'labels': 3567, 'matched': 3567, 'unmatched_log_lines': 19, 'unmatched_labels': 0}


camp7 JS {'log_lines': 6372, 'labels': 6372, 'matched': 6372, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


camp7 DVWA {'log_lines': 6456, 'labels': 6387, 'matched': 6387, 'unmatched_log_lines': 69, 'unmatched_labels': 0}
450 flagged sessions; 0 of them normal


truth
enum                         53
xss                          52
sqli                         52
traversal                    47
scan                         46
cmdi                         44
bruteforce                   36
scan+sqli+xss                 6
enum+scan+sqli+xss            6
bruteforce+enum+scan+sqli     6
Name: count, dtype: int64

## 1. Choosing `eps` on one site, scoring on the other

`eps` is the distance under which two sessions count as neighbours. To avoid tuning it on the data we score, it is chosen on one site and then used on the other site.

In [2]:
EPS = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]
def ari(t, eps):
    return adjusted_rand_score(t['truth'], DBSCAN(eps=eps, min_samples=5).fit_predict(t[STEPS].values))

by_site = {s: T[T['site'] == s] for s in ['Juice Shop', 'DVWA']}
grid = pd.DataFrame({s: [ari(t, e) for e in EPS] for s, t in by_site.items()}, index=EPS).round(3)
grid.index.name = 'eps'
rows = []
for chosen_on, scored_on in [('Juice Shop', 'DVWA'), ('DVWA', 'Juice Shop')]:
    best = grid[chosen_on].idxmax()
    t = by_site[scored_on]
    rows.append({'eps chosen on': chosen_on, 'eps': best, 'scored on': scored_on,
                 'DBSCAN ARI': round(ari(t, best), 3),
                 'baseline ARI (main step)': round(adjusted_rand_score(t['truth'], t[STEPS].idxmax(axis=1)), 3)})
display(grid)
pd.DataFrame(rows)

,Juice Shop,DVWA
eps,,
0.05,0.542,0.332
0.10,0.542,0.332
0.15,0.627,0.417
0.20,0.873,0.704
0.25,0.894,0.594
0.30,0.753,0.602
0.40,0.610,0.297


,eps chosen on,eps,scored on,DBSCAN ARI,baseline ARI (main step)
0,Juice Shop,0.25,DVWA,0.594,0.511
1,DVWA,0.20,Juice Shop,0.873,0.674


## 2. All sessions of both sites together

With `eps = 0.2` (the value chosen on DVWA above): which groups come out, how pure they are, and whether each group holds attackers from both sites.

In [3]:
T['group'] = DBSCAN(eps=0.2, min_samples=5).fit_predict(T[STEPS].values)
groups = T.groupby('group').agg(
    sessions=('truth', 'size'),
    main_truth=('truth', lambda s: s.value_counts().index[0]),
    purity=('truth', lambda s: round(s.value_counts().iloc[0] / len(s), 2)),
    juice_shop=('site', lambda s: int((s == 'Juice Shop').sum())),
    dvwa=('site', lambda s: int((s == 'DVWA').sum())))
print('ARI all sessions:', round(adjusted_rand_score(T['truth'], T['group']), 3),
      '| baseline:', round(adjusted_rand_score(T['truth'], T[STEPS].idxmax(axis=1)), 3))
groups

ARI all sessions: 0.792 | baseline: 0.612


,sessions,main_truth,purity,juice_shop,dvwa
group,,,,,
-1,23,bruteforce+cmdi+sqli+traversal,0.17,11,12
0,34,cmdi,1.00,23,11
1,80,enum,0.66,49,31
2,47,traversal,1.00,29,18
3,46,scan,1.00,29,17
4,52,xss,1.00,31,21
5,52,sqli,1.00,32,20
6,36,bruteforce,1.00,23,13
7,10,cmdi,1.00,7,3


Group `-1` is DBSCAN's noise (sessions with no close neighbours).

## Conclusion

- No normal session is flagged, so all groups are attackers.
- Every group that DBSCAN forms around **one** kind of attack is pure (purity 1.00), and every group holds attackers from **both** sites: DBSCAN groups attackers by what they did, not by IP or by site.
- With `eps` chosen on the other site, DBSCAN beats the main-step baseline on both sites (see table 1).
- Weak points: campaign sessions (several attack kinds) land in mixed groups, ID enumeration groups together with campaigns that enumerate, and command injection splits into two groups.
- Limits: all sessions come from our generator, which gives every session its own IP, so we can show "attackers that behave alike", not "the same person behind several IPs".
- Decision: DBSCAN stays an **experiment** in the report (the rule agreed before the experiment: it would go into the system only if clearly useful). A "similar attackers" view in the frontend is possible later, as display only.